In [1]:
import sys
print(sys.executable)
print(sys.version)

/opt/conda/envs/mlops/bin/python
3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:21:38) [GCC 14.3.0]


In [2]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet"
file_path = data_dir / "yellow_tripdata_2021-01.parquet"

if not file_path.exists():
    urlretrieve(url, file_path)

file_path

PosixPath('../data/yellow_tripdata_2021-01.parquet')

In [3]:
import pandas as pd

df = pd.read_parquet(file_path)
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2021-01-01 00:30:10,2021-01-01 00:36:12,1.0,2.10,1.0,N,142,43,2,8.0,3.0,0.5,0.00,0.0,0.3,11.80,2.5,NaN
1,1,2021-01-01 00:51:20,2021-01-01 00:52:19,1.0,0.20,1.0,N,238,151,2,3.0,0.5,0.5,0.00,0.0,0.3,4.30,0.0,NaN
2,1,2021-01-01 00:43:30,2021-01-01 01:11:06,1.0,14.70,1.0,N,132,165,1,42.0,0.5,0.5,8.65,0.0,0.3,51.95,0.0,NaN
3,1,2021-01-01 00:15:48,2021-01-01 00:31:01,0.0,10.60,1.0,N,138,132,1,29.0,0.5,0.5,6.05,0.0,0.3,36.35,0.0,NaN
4,2,2021-01-01 00:31:49,2021-01-01 00:48:21,1.0,4.94,1.0,N,68,33,1,16.5,0.5,0.5,4.06,0.0,0.3,24.36,2.5,NaN


In [4]:
df.shape

(1369769, 19)

In [5]:
df.dtypes

VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
dtype: object

In [6]:
df[[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID"
]].head()

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID
0,2021-01-01 00:30:10,2021-01-01 00:36:12,142,43
1,2021-01-01 00:51:20,2021-01-01 00:52:19,238,151
2,2021-01-01 00:43:30,2021-01-01 01:11:06,132,165
3,2021-01-01 00:15:48,2021-01-01 00:31:01,138,132
4,2021-01-01 00:31:49,2021-01-01 00:48:21,68,33


In [7]:
urls = {
    "2021-01": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet",
    "2021-02": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-02.parquet",
}

for month, url in urls.items():
    destination = data_dir / f"yellow_tripdata_{month}.parquet"
    if not destination.exists():
        urlretrieve(url, destination)

In [8]:
sorted(path.name for path in data_dir.glob("yellow_tripdata_*.parquet"))

['yellow_tripdata_2021-01.parquet', 'yellow_tripdata_2021-02.parquet']

In [9]:
files = sorted(data_dir.glob("yellow_tripdata_*.parquet"))
frames = []

for parquet_file in files:
    frame = pd.read_parquet(parquet_file)
    month = parquet_file.stem.removeprefix("yellow_tripdata_")
    frame["source_month"] = month
    frames.append(frame)

trips = pd.concat(frames, ignore_index=True)

trips.groupby("source_month").size()

source_month
2021-01    1369769
2021-02    1371709
dtype: int64

In [10]:
total_sources = sum(len(frame) for frame in frames)
total_trips = len(trips)

print("Total des fichiers :", total_sources)
print("Total après concaténation :", total_trips)

assert total_trips == total_sources, "Le nombre de lignes ne correspond pas."
print("Vérification réussie.")

Total des fichiers : 2741478
Total après concaténation : 2741478
Vérification réussie.
